In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os
import kagglehub
import matplotlib.pyplot  as plt
from PIL import Image

# Download latest version
path = kagglehub.dataset_download("fahadullaha/facial-emotion-recognition-dataset")
print("Path to dataset files:", path)
sub_dir = os.path.join(path, 'processed_data')

Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\fahadullaha\facial-emotion-recognition-dataset\versions\1


## Simple CNN


In [2]:

transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


In [3]:

if os.path.exists(sub_dir):
    full_dataset = datasets.ImageFolder(root=sub_dir, transform=transform)
else:
    full_dataset = datasets.ImageFolder(root=path, transform=transform)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print("Tapılan Siniflər:", full_dataset.classes)
print("Cəmi şəkil sayı:", len(full_dataset))


Tapılan Siniflər: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Cəmi şəkil sayı: 49779


In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 7) 
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

cuda


In [ ]:
epochs = 20
print(f"Training started on {device}...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()           
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

Training started on cuda...
Epoch 1/20, Loss: 1.6539
Epoch 2/20, Loss: 1.4050
Epoch 3/20, Loss: 1.2888
Epoch 4/20, Loss: 1.2090
Epoch 5/20, Loss: 1.1590
Epoch 6/20, Loss: 1.1235
Epoch 7/20, Loss: 1.0881
Epoch 8/20, Loss: 1.0660
Epoch 9/20, Loss: 1.0404
Epoch 10/20, Loss: 1.0180
Epoch 11/20, Loss: 1.0003
Epoch 12/20, Loss: 0.9869
Epoch 13/20, Loss: 0.9751
Epoch 14/20, Loss: 0.9608
Epoch 15/20, Loss: 0.9489
Epoch 16/20, Loss: 0.9361
Epoch 17/20, Loss: 0.9228
Epoch 18/20, Loss: 0.9120
Epoch 19/20, Loss: 0.9004
Epoch 20/20, Loss: 0.8950


In [6]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _,predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 65.33%


In [7]:
import math

def plot_grid(folder_path, model, n=14, cols=5):
    images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png'))][:n]
    rows = math.ceil(len(images) / cols)
    plt.figure(figsize=(cols * 4, rows * 4)) 
    
    model.eval()
    class_names = full_dataset.classes

    for i, name in enumerate(images):
        img = Image.open(os.path.join(folder_path, name))
        
        t = transforms.Compose([
            transforms.Resize((48, 48)), 
            transforms.Grayscale(1), 
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        tensor = t(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred = class_names[model(tensor).argmax(1).item()]

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img, cmap='gray' if img.mode == 'L' else None)
        plt.title(f"P: {pred}", fontsize=15, color='blue')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

test_path = r"C:\Users\USER\Documents\PYTHON\MYDL\CNN\Facial Emotion Recognition Dataset\test_pics"
if os.path.exists(test_path):
    plot_grid(test_path, model)


## ResNet

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)), 
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
full_dataset = datasets.ImageFolder(root=sub_dir if os.path.exists(sub_dir) else path, transform=transform)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Siniflər: {full_dataset.classes}")
print(f"Cəmi şəkil: {len(full_dataset)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_resnet():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    

    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 7)
    )
    return model.to(device)

model = build_resnet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001) 

In [ ]:
epochs = 10 
print(f"Cihaz: {device}. Training başlayır...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [ ]:
def plot_grid_results(folder_path, model, n=14, cols=5):

    images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png'))][:n]
    rows = math.ceil(len(images) / cols)
    plt.figure(figsize=(cols * 4, rows * 4))
    
    model.eval()
    class_names = full_dataset.classes

    for i, name in enumerate(images):
        img_path = os.path.join(folder_path, name)
        img = Image.open(img_path)
        
        t = transforms.Compose([
            transforms.Resize((64, 64)), 
            transforms.Grayscale(1), 
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        tensor = t(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred_idx = model(tensor).argmax(1).item()
            prediction = class_names[pred_idx]

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img, cmap='gray' if img.mode == 'L' else None)
        plt.title(f"Pred: {prediction}", fontsize=15, color='darkgreen')
        plt.axis('off')
   
    plt.tight_layout()
    plt.show()

test_path = r"C:\Users\USER\Documents\PYTHON\MYDL\CNN\Facial Emotion Recognition Dataset\test_pics"
if os.path.exists(test_path):
    plot_grid_results(test_path, model)